In [48]:
import polars as pl
import catboost as cb
import os

path_train = os.path.join(os.getcwd(), "..", "data","train_main_features.parquet")
path_test = os.path.join(os.getcwd(),"..","data","test_main_features.parquet")
path_target = os.path.join(os.getcwd(),"..","data","train_target.parquet")
train = pl.read_parquet(path_train)
test = pl.read_parquet(path_test)
target = pl.read_parquet(path_target)

print(train.head(5))

target_columns = [c for c in target.columns if c != "customer_id"]


shape: (5, 200)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ customer_ ┆ cat_featu ┆ cat_featu ┆ cat_featu ┆ … ┆ num_featu ┆ num_featu ┆ num_featu ┆ num_feat │
│ id        ┆ re_1      ┆ re_2      ┆ re_3      ┆   ┆ re_129    ┆ re_130    ┆ re_131    ┆ ure_132  │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ i32       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1000001   ┆ 1.0       ┆ 0.0       ┆ 2.0       ┆ … ┆ -0.107666 ┆ -0.418616 ┆ null      ┆ null     │
│ 1000002   ┆ 1.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ -0.170724 ┆ -0.805771 ┆ -0.397803 ┆ -0.37373 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 4        │
│ 1000003   ┆ 1.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ -0.170724 ┆ -0.602005

In [49]:
target_summ = []
for c in target_columns:
    vals = target.select(pl.col(c).drop_nulls().unique().sort()).to_series().to_list()
    target_summ.append({
        "target": c,
        "n_unique": len(vals),
        "unique_values_preview": vals[:10]
    })

print(pl.DataFrame(target_summ))

shape: (41, 3)
┌─────────────┬──────────┬───────────────────────┐
│ target      ┆ n_unique ┆ unique_values_preview │
│ ---         ┆ ---      ┆ ---                   │
│ str         ┆ i64      ┆ list[f64]             │
╞═════════════╪══════════╪═══════════════════════╡
│ target_1_1  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_2  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_3  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_4  ┆ 2        ┆ [0.0, 1.0]            │
│ target_1_5  ┆ 2        ┆ [0.0, 1.0]            │
│ …           ┆ …        ┆ …                     │
│ target_9_5  ┆ 2        ┆ [0.0, 1.0]            │
│ target_9_6  ┆ 2        ┆ [0.0, 1.0]            │
│ target_9_7  ┆ 2        ┆ [0.0, 1.0]            │
│ target_9_8  ┆ 2        ┆ [0.0, 1.0]            │
│ target_10_1 ┆ 2        ┆ [0.0, 1.0]            │
└─────────────┴──────────┴───────────────────────┘


In [50]:
app = []
for c in train.columns:
    if c.startswith("n"):
        tr_sl_max = train.select(
            pl.col(c).max()
        ).item()
        tr_sl_min = train.select(
            pl.col(c).min()
        ).item()
        app.append({
            "max_value": tr_sl_max,
            "min_value": tr_sl_min,
            "column": c
        })
max_val = max(x["max_value"] for x in app)
min_val = min(x["min_value"] for x in app)
print(max_val, min_val)

1440.2291939697725 -10.039640917881606


In [51]:
for c in train.columns:
    exists = train.select((pl.col(c) == 2141.0).any()).item()
    if exists:
        print(c)


cat_feature_39


In [52]:

print(target.select(
    pl.col("target_9_8")
    .filter((pl.col("target_9_8") != 0.0) & (pl.col("target_9_8") != 1.0))
    .unique()
    .sort()
    )
)

shape: (0, 1)
┌────────────┐
│ target_9_8 │
│ ---        │
│ f64        │
╞════════════╡
└────────────┘


In [53]:
iqr_summary = []
for c in train.columns:
        if c.startswith("n"):
            q1 = train.select(pl.col(c).quantile(0.25)).item()
            q3 = train.select(pl.col(c).quantile(0.75)).item()
            if q1 is None or q3 is None:
                continue

            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr

            sort = train.select(
                ((pl.col(c) < lower) | (pl.col(c) > upper)).sum()
            ).item()

            iqr_summary.append({
                "column": c,
                "q1": q1,
                "q3": q3,
                "iqr": iqr,
                "lower_bound": lower,
                "upper_bound": upper,
                "outer_count": sort
            })

iqr_df = pl.DataFrame(iqr_summary).sort("outer_count", descending=True)
print(iqr_df)

shape: (132, 7)
┌─────────────────┬───────────┬───────────┬──────────┬─────────────┬─────────────┬─────────────┐
│ column          ┆ q1        ┆ q3        ┆ iqr      ┆ lower_bound ┆ upper_bound ┆ outer_count │
│ ---             ┆ ---       ┆ ---       ┆ ---      ┆ ---         ┆ ---         ┆ ---         │
│ str             ┆ f64       ┆ f64       ┆ f64      ┆ f64         ┆ f64         ┆ i64         │
╞═════════════════╪═══════════╪═══════════╪══════════╪═════════════╪═════════════╪═════════════╡
│ num_feature_98  ┆ -0.004499 ┆ -0.004499 ┆ 0.0      ┆ -0.004499   ┆ -0.004499   ┆ 171499      │
│ num_feature_21  ┆ -0.140323 ┆ -0.124331 ┆ 0.015993 ┆ -0.164312   ┆ -0.100342   ┆ 137749      │
│ num_feature_67  ┆ -0.430568 ┆ -0.430568 ┆ 0.0      ┆ -0.430568   ┆ -0.430568   ┆ 123662      │
│ num_feature_76  ┆ -0.0279   ┆ -0.02019  ┆ 0.00771  ┆ -0.039465   ┆ -0.008625   ┆ 123463      │
│ num_feature_62  ┆ -0.025496 ┆ -0.017985 ┆ 0.007512 ┆ -0.036764   ┆ -0.006717   ┆ 114475      │
│ …           

In [54]:
train_new = train.select(
    pl.col("num_feature_98").n_unique()
)
print(train_new)
print(train["num_feature_98"].count())

shape: (1, 1)
┌────────────────┐
│ num_feature_98 │
│ ---            │
│ u32            │
╞════════════════╡
│ 859            │
└────────────────┘
746785


In [55]:
# app = []
# for c in train.columns:
#     tr = train.select(
#         (pl.col(c).is_null()).sum()
#     ).item()
#     if tr > 700000:
#         app.append({
#             "count": tr,
#             "column": c
#         })
# app_df = pl.DataFrame(app)

# cols_to_drop = app_df["column"].to_list()
# train_without_extra = train.clone().drop(cols_to_drop)


# print(train_without_extra.shape)

In [56]:
#find rows with count Nan more than 0.95
clear_train =train.select(pl.exclude("customer_id"))
n_rows = clear_train.height
nan_info = []
for c in clear_train.columns:
    len_nun = clear_train.select((pl.col(c).is_null()).sum().sort()).item()
    nan_info.append({
        "column": c,
        "count Nan": len_nun
    })
stats = pl.DataFrame(nan_info)

stats = stats.with_columns(
    (pl.col("count Nan") / n_rows).alias("null_ratio")
).sort("null_ratio", descending=True)

#print(stats.head(20))

threshold = 0.95

cols_to_drop = (
    stats
    .filter(pl.col("null_ratio") > threshold)
    .get_column("column")
    .to_list()
)

train_without_extra = clear_train.drop(cols_to_drop)
print(train_without_extra.shape)


(750000, 181)


In [57]:
for c in train_without_extra.columns:
    if c.startswith("cat"):
        train_without_extra = train_without_extra.with_columns([
            pl.col(c).cast(pl.String).cast(pl.Categorical)
        ])

print(train_without_extra)

shape: (750_000, 181)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ cat_featu ┆ cat_featu ┆ cat_featu ┆ cat_featu ┆ … ┆ num_featu ┆ num_featu ┆ num_featu ┆ num_feat │
│ re_1      ┆ re_2      ┆ re_3      ┆ re_4      ┆   ┆ re_129    ┆ re_130    ┆ re_131    ┆ ure_132  │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ cat       ┆ cat       ┆ cat       ┆ cat       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1.0       ┆ 0.0       ┆ 2.0       ┆ 1.0       ┆ … ┆ -0.107666 ┆ -0.418616 ┆ null      ┆ null     │
│ 1.0       ┆ 0.0       ┆ 0.0       ┆ 1.0       ┆ … ┆ -0.170724 ┆ -0.805771 ┆ -0.397803 ┆ -0.37373 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 4        │
│ 1.0       ┆ 0.0       ┆ 0.0       ┆ 1.0       ┆ … ┆ -0.170724 ┆ -0.

In [ ]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier


X = train_without_extra.to_pandas()
y = target.drop("customer_id").to_pandas()
y = y["target_1_1"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size = 0.2, random_state= 42
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full
)

In [59]:
print(X_train.shape)
print(y_train.shape)

(450000, 181)
(450000,)


In [67]:
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
cat_features = [x for x in train_without_extra.columns if x.startswith("cat")]

catboost = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=6,
    cat_features=cat_features,
    loss_function='Logloss',
    verbose=50,
    l2_leaf_reg = 3,
    eval_metric='AUC',
    task_type='GPU',
    early_stopping_rounds=100
)

xgboost = XGBClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    tree_method='hist',
    enable_categorical=True,
    early_stopping_rounds=100
)

catboost.fit(X_train, y_train, eval_set=(X_valid, y_valid))
pred1 = catboost.predict_proba(X_test)[:, 1]

xgboost.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)
pred2 = xgboost.predict_proba(X_test)[:, 1]

final_pred = (pred1 + pred2) /2
print("AUC:", roc_auc_score(y_test, final_pred))

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6003433	best: 0.6003433 (0)	total: 125ms	remaining: 6m 14s
50:	test: 0.8187306	best: 0.8187306 (50)	total: 6.35s	remaining: 6m 7s
100:	test: 0.8456974	best: 0.8456974 (100)	total: 12.5s	remaining: 5m 59s
150:	test: 0.8557023	best: 0.8557023 (150)	total: 18.7s	remaining: 5m 53s
200:	test: 0.8610448	best: 0.8610448 (200)	total: 24.9s	remaining: 5m 46s
250:	test: 0.8635994	best: 0.8635994 (250)	total: 31s	remaining: 5m 39s
300:	test: 0.8654821	best: 0.8654821 (300)	total: 37.1s	remaining: 5m 32s
350:	test: 0.8668406	best: 0.8668406 (350)	total: 43.1s	remaining: 5m 25s
400:	test: 0.8684771	best: 0.8684771 (400)	total: 49s	remaining: 5m 17s
450:	test: 0.8698326	best: 0.8698326 (450)	total: 55s	remaining: 5m 10s
500:	test: 0.8705367	best: 0.8705367 (500)	total: 1m	remaining: 5m 4s
550:	test: 0.8714380	best: 0.8714421 (549)	total: 1m 6s	remaining: 4m 57s
600:	test: 0.8722374	best: 0.8722573 (593)	total: 1m 12s	remaining: 4m 50s
650:	test: 0.8727832	best: 0.8727832 (650)	total: 1m 1